# Figure 20 -- multi-GPU reconstruction scaling

Loads `bench/results/density_reconstruction/multigpu_scaling.json`, produced by `bench/payoff_static/multigpu_scaling.py`. No computation here.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == "jaccpot_paper" else pathlib.Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from examples.jaccpot_paper.common import jsonio, style

style.apply()
FIG_DIR = jsonio.RESULTS_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
art = jsonio.read_result("density_reconstruction/multigpu_scaling.json")
cfg, data = art["config"], art["data"]
recs = [r for r in data["records"] if not r.get("failed") and not r.get("skipped")]
ceiling = data.get("ceiling", [])
if not recs:
    raise SystemExit("multigpu_scaling.json has no successful rows")

fig, axes = style.figure(width=style.TWO_COL, height=2.9, ncols=2)

# -- left: strong scaling at fixed parameter count ------------------------- #
ax = axes[0]
sel = sorted(recs, key=lambda r: r["num_devices"])
ndev = [r["num_devices"] for r in sel]
wall = [r["timing"]["wall_seconds"] for r in sel]
ax.plot(ndev, wall, marker=style.MARKERS[0], color=style.ENTITY["gpu"],
        label="measured")
ax.plot(ndev, [wall[0] * ndev[0] / d for d in ndev], ls=":",
        color=style.INK_MUTED, label="ideal")
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xlabel("devices"); ax.set_ylabel("fit wall-clock [s]")
style.finish(ax, legend=True, legend_kwargs={"loc": "lower left", "fontsize": 5.6})

# -- right: the ceiling, measured by running until it broke ---------------- #
ax = axes[1]
if ceiling:
    ok, bad = {}, {}
    for r in ceiling:
        target = bad if r.get("failed") else ok
        target.setdefault(r["num_devices"], []).append(r["num_free_parameters"])
    devices = sorted(set(ok) | set(bad))
    largest = [max(ok.get(d, [0])) for d in devices]
    ax.plot(devices, largest, marker=style.MARKERS[1], color=style.ENTITY["gpu"],
            label="largest $P$ that ran")
    for d in devices:
        for p in bad.get(d, []):
            ax.scatter([d], [p], marker="x", s=26, color=style.CATEGORICAL[2],
                       label="out of memory" if d == devices[0] else None)
    ax.set_xscale("log", base=2)
    ax.set_xlabel("devices")
else:
    ax.text(0.5, 0.5, "no ceiling ladder in this artifact", ha="center",
            va="center", fontsize=6, color=style.INK_MUTED,
            transform=ax.transAxes)
    ax.set_xlabel("devices")
ax.set_yscale("log"); ax.set_ylabel("free parameters $P$")
style.finish(ax, legend=bool(ceiling), legend_kwargs={"loc": "upper left",
                                                      "fontsize": 5.6})

fig.tight_layout()
style.footer(fig, "%s, %s sharding, N=%d, %d iterations, leaf %d" % (
    art["meta"]["device_kind"], cfg.get("sharding_mode", "?"), cfg["n"][0],
    cfg["iterations"], cfg["leaf_size"]))
style.save(fig, str(FIG_DIR / "fig20_multigpu_reconstruction_scaling.pdf"))


## Caption


The reconstruction across multiple devices. **Left:** wall-clock for the same
fit on an increasing device count, at fixed parameter count, against ideal
scaling -- to be read against the strong- and weak-scaling figures of Sect.~5.
**Right:** the largest free-parameter count that ran, per device count, with the
configurations that exhausted memory marked. The ceiling is measured by climbing
a ladder until it broke, not inferred from a memory model. This is *parameter*
sharding: the optimisation's parameter array is distributed across the mesh, so
the parameter count is bounded by aggregate device memory rather than by one
device's. It is not the distributed force evaluation of Sect.~5, which partitions
the sources and exchanges halos; the two are not interchangeable and each run's
artifact records which it used.
